In [1]:
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou
from pytubefix import YouTube
from pathlib import Path
import cv2
import os
import shutil
import torch
import glob
import logging

print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
### Configs

detect_model = YOLO("models/11m.pt")
pose_model = YOLO("yolo11m-pose.pt")

min_conf = 0.5
input_directory = Path("video/input")
output_directory = Path("video/output")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = 30


rgb = (0, 0, 255)
thickness = 1
font_scale = 0.6

In [30]:
### Run Model

writer = None

output_directory.mkdir(parents=True, exist_ok=True)
for file in output_directory.glob("*.mp4"):
    file.unlink()

first = True
frame_count = 0

for video_index, video in enumerate(input_directory.glob("*.mp4")):

    print(f"Processing: {video}")
    
    with open(f"logs/log_{video_index}.txt", "w") as log_file:

        log_file.write(f"Processing: {video}")

        # Entity Detection
        detect_results = detect_model.track(
            source=video,
            device=0,
            half=True,
            stream=True,
            tracker="botsort.yaml"
        )

        # Frame Iteration
        for result in detect_results:

            # Get Frame and Boxes
            frame = result.orig_img.copy()
            boxes = result.boxes
            if boxes is None or len(boxes) == 0:
                continue
            
            # Get tracking IDs
            track_ids = (
                boxes.id.int().cpu().tolist()
                if boxes.id is not None
                else [None] * len(boxes)
            )

            # Assign New Writer
            if writer is None:
                height, width = frame.shape[:2]
                output_file = output_directory / f"{video.stem}.mp4"
                writer = cv2.VideoWriter(
                    str(output_file),
                    fourcc,
                    30,
                    (width, height)
                )
            
            # Iterate Each Frame for all Entities
            for box, cls_id, conf, track_id in zip(
                boxes.xyxy,
                boxes.cls,
                boxes.conf,
                track_ids
            ):
                # Frame Drawing
                x1, y1, x2, y2 = map(int, box)
                label = f"ID:{track_id} CLS:{detect_model.names[int(cls_id)]} CONF:{conf:.2f}"
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    rgb,
                    thickness
                )
                cv2.rectangle(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    rgb,
                    thickness
                )
            
            # Write Annotated Frame
            writer.write(frame)
    
        log_file.write("Done")
    
        writer.release()
        writer = None

print("All Done")









Processing: video\input\clip2.mp4

video 1/1 (frame 1/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 67.9ms
video 1/1 (frame 2/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 68.4ms
video 1/1 (frame 3/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 54.2ms
video 1/1 (frame 4/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 55.7ms
video 1/1 (frame 5/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 52.8ms
video 1/1 (frame 6/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 52.9ms
video 1/1 (frame 7/3922) c:\Gabriel_Files\Programming_Files\School\Thesis\src\video\input\clip2.mp4: 512x480 2 persons, 2 phones, 26.2ms
video 